# RelBench rel-avito Dataset Preparation for JRN Experiments

This notebook demonstrates the transformation of raw RelBench rel-avito data into a unified dataset format suitable for Join Reproduction Number (JRN) experiments.

**What it does:**
- Loads the curated mini dataset (`mini_demo_data.json`) containing representative examples
- Explores 4 item types: dataset metadata, table data, FK join metadata, task splits
- Visualizes table sizes, foreign-key fanout distributions, and task characteristics
- Shows the relational schema of the Avito ad marketplace (8 tables, 11 FK relationships)

**Source:** RelBench rel-avito (Avito ad marketplace) relational database, arXiv:2407.20060

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

# Load the curated mini dataset
with open("mini_demo_data.json", "r") as f:
    data = json.load(f)

# Categorize items by type
type_counts = Counter(item["type"] for item in data)
print(f"Total items: {len(data)}")
print(f"Item types: {dict(type_counts)}")
for t, c in type_counts.items():
    print(f"  - {t}: {c} items")

## 1. Dataset Metadata Overview

The first item contains global metadata about the RelBench rel-avito database: table names, row counts, foreign key structure, and prediction tasks.

In [ ]:
# Extract dataset metadata
meta = [item for item in data if item["type"] == "dataset_metadata"][0]

print(f"Dataset: {meta['dataset_name']}")
print(f"Source:  {meta['source']}")
print(f"Paper:   {meta['paper']}")
print(f"License: {meta['license']}")
print(f"\nNum tables: {meta['num_tables']}")
print(f"Table names: {meta['table_names']}")
print(f"Total rows across tables: {meta['total_rows_across_tables']:,}")
print(f"Num FK relationships: {meta['num_fk_relationships']}")
print(f"Prediction tasks: {meta['task_names']}")
print(f"\nVal timestamp:  {meta['val_timestamp']}")
print(f"Test timestamp: {meta['test_timestamp']}")

In [ ]:
# Visualize table sizes from metadata
table_summary = meta["table_summary"]
table_names = list(table_summary.keys())
row_counts = [table_summary[t]["num_rows"] for t in table_names]

fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(table_names)))
bars = ax.barh(table_names, row_counts, color=colors)
ax.set_xlabel("Number of Rows")
ax.set_title("RelBench rel-avito: Table Sizes")
ax.set_xscale("log")

# Add value labels
for bar, count in zip(bars, row_counts):
    ax.text(bar.get_width() * 1.1, bar.get_y() + bar.get_height()/2,
            f"{count:,}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

## 2. Relational Schema: Foreign Key Graph

The schema has 11 FK relationships connecting 8 tables. Below we display the FK edges and show which tables serve as hubs in the relational graph.

In [ ]:
# Build FK edge list from table metadata
fk_edges = []
for tname, tinfo in table_summary.items():
    for fk_col, target_table in tinfo.get("fkeys", {}).items():
        fk_edges.append((tname, fk_col, target_table))

print(f"Foreign key relationships ({len(fk_edges)} edges):")
print("-" * 60)
fk_df = pd.DataFrame(fk_edges, columns=["Source Table", "FK Column", "Target Table"])
print(fk_df.to_string(index=False))

# Count how many FK edges reference each table (in-degree = how many tables point to it)
target_counts = Counter(e[2] for e in fk_edges)
source_counts = Counter(e[0] for e in fk_edges)
print(f"\nMost-referenced tables (FK targets):")
for t, c in target_counts.most_common():
    print(f"  {t}: referenced by {c} FK column(s)")

## 3. FK Join Metadata: Fanout Distributions

Each FK relationship has a fanout distribution — how many child rows reference each parent. High fanout and skewed distributions are key signals for JRN complexity.

In [ ]:
# Extract FK join metadata items
fk_joins = [item for item in data if item["type"] == "fk_join_metadata"]

print(f"FK join metadata items: {len(fk_joins)}")
print("=" * 80)

rows = []
for fk in fk_joins:
    label = f"{fk['source_table']}.{fk['source_fk_col']} → {fk['target_table']}"
    rows.append({
        "Join": label,
        "Edges": f"{fk['num_edges']:,}",
        "Coverage": f"{fk['join_coverage']:.1%}",
        "Fanout Mean": f"{fk['fanout_mean']:.1f}",
        "Fanout Median": f"{fk['fanout_median']:.0f}",
        "Fanout Max": f"{fk['fanout_max']:,}",
        "Card. Ratio": f"{fk['cardinality_ratio']:.1f}",
    })

join_df = pd.DataFrame(rows)
print(join_df.to_string(index=False))

In [ ]:
# Visualize fanout distributions across FK joins
fig, axes = plt.subplots(1, len(fk_joins), figsize=(5 * len(fk_joins), 4), sharey=False)
if len(fk_joins) == 1:
    axes = [axes]

for ax, fk in zip(axes, fk_joins):
    label = f"{fk['source_table']}.{fk['source_fk_col']}\n→ {fk['target_table']}"
    percentiles = [fk["fanout_min"], fk["fanout_p25"], fk["fanout_median"],
                   fk["fanout_p75"], fk["fanout_p95"], fk["fanout_max"]]
    pct_labels = ["min", "p25", "median", "p75", "p95", "max"]

    colors_bar = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(percentiles)))
    ax.bar(pct_labels, percentiles, color=colors_bar, edgecolor="black", linewidth=0.5)
    ax.set_title(label, fontsize=10)
    ax.set_ylabel("Fanout")
    ax.set_yscale("log")
    ax.tick_params(axis="x", rotation=45)

fig.suptitle("FK Join Fanout Distributions (log scale)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 4. Table Data: Sample Rows and Column Statistics

Each `table_data` item contains column metadata (dtypes, missing rates, summary statistics) and a small sample of actual rows. This lets us understand the data without loading the full tables.

In [ ]:
# Extract table data items and display sample rows
table_items = [item for item in data if item["type"] == "table_data"]

for tbl in table_items:
    print(f"\n{'='*70}")
    print(f"Table: {tbl['table_name']}  ({tbl['num_rows']:,} rows × {tbl['num_cols']} cols)")
    print(f"Primary key: {tbl['primary_key_col']}   Time col: {tbl['time_col']}")
    if tbl['foreign_keys']:
        print(f"Foreign keys: {tbl['foreign_keys']}")
    if tbl.get('subsampled'):
        print(f"Subsampled: {tbl['subsample_size']:,} / {tbl['full_size']:,}")
    print(f"\nSample rows:")
    sample_df = pd.DataFrame(tbl["data"])
    print(sample_df.to_string(index=False))
    
    # Show missing rates for columns with > 0 missing
    missing = {k: v for k, v in tbl["missing_rates"].items() if v > 0}
    if missing:
        print(f"\nColumns with missing data:")
        for col, rate in missing.items():
            print(f"  {col}: {rate:.2%}")

In [ ]:
# Visualize column dtype composition per table
fig, ax = plt.subplots(figsize=(10, 4))

dtype_categories = {"Int64": "integer", "int64": "integer", "float64": "float",
                    "object": "string", "datetime64[ns]": "datetime"}

table_dtype_data = {}
for tbl in table_items:
    dtype_counts = Counter()
    for col, dt in tbl["column_dtypes"].items():
        cat = dtype_categories.get(dt, dt)
        dtype_counts[cat] += 1
    table_dtype_data[tbl["table_name"]] = dtype_counts

all_cats = sorted(set(c for d in table_dtype_data.values() for c in d))
cat_colors = {"integer": "#4C72B0", "float": "#55A868", "string": "#C44E52",
              "datetime": "#8172B2"}

x = np.arange(len(table_dtype_data))
width = 0.15
for i, cat in enumerate(all_cats):
    vals = [table_dtype_data[t].get(cat, 0) for t in table_dtype_data]
    ax.bar(x + i * width, vals, width, label=cat,
           color=cat_colors.get(cat, "#999999"), edgecolor="black", linewidth=0.5)

ax.set_xticks(x + width * (len(all_cats) - 1) / 2)
ax.set_xticklabels(table_dtype_data.keys(), rotation=30, ha="right")
ax.set_ylabel("Number of Columns")
ax.set_title("Column Data Types per Table")
ax.legend(title="Dtype")
plt.tight_layout()
plt.show()

## 5. Prediction Tasks: Task Splits

RelBench defines 4 prediction tasks over the Avito database — regression, binary classification, and link prediction. Each task has temporal train/val/test splits.

In [ ]:
# Extract task split items
tasks = [item for item in data if item["type"] == "task_split"]

print(f"Task splits: {len(tasks)}")
print("=" * 80)

for task in tasks:
    print(f"\nTask: {task['task_name']}")
    print(f"  Type: {task['task_type']}")
    print(f"  Fold: {task['metadata_fold']}")
    print(f"  Samples: {task['num_samples']:,}")
    print(f"  Entity table: {task.get('entity_table', 'N/A')}")
    print(f"  Target col: {task.get('target_col', 'N/A')}")
    print(f"  Time range: {task['timestamp_min']} → {task['timestamp_max']}")
    print(f"  Timedelta: {task['timedelta_days']} days")
    
    if task['task_type'] == 'regression' and 'target_stats' in task:
        stats = task['target_stats']
        print(f"  Target stats: mean={stats['mean']:.4f}, std={stats['std']:.4f}, "
              f"min={stats['min']:.4f}, max={stats['max']:.4f}")
    elif task['task_type'] == 'binary_classification' and 'positive_rate' in task:
        print(f"  Positive rate: {task['positive_rate']:.2%}")
        print(f"  Label counts: {task['label_counts']}")
    elif task['task_type'] == 'link_prediction':
        print(f"  Columns: {task['columns']}")
    
    # Show sample data
    print(f"  Sample data ({len(task['data'])} rows):")
    for row in task["data"][:2]:
        row_display = {k: (v if not isinstance(v, list) else f"[{len(v)} items]") for k, v in row.items()}
        print(f"    {row_display}")

In [ ]:
# Visualize task characteristics
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: Task sample sizes
task_names_short = [t["task_name"].split("/")[-1] for t in tasks]
task_sizes = [t["num_samples"] for t in tasks]
task_types = [t["task_type"] for t in tasks]
type_colors = {"regression": "#E69F00", "binary_classification": "#56B4E9",
               "link_prediction": "#009E73"}

bar_colors = [type_colors.get(tt, "#999") for tt in task_types]
bars = axes[0].bar(task_names_short, task_sizes, color=bar_colors, edgecolor="black", linewidth=0.5)
axes[0].set_ylabel("Number of Samples")
axes[0].set_title("Task Sizes (Training Split)")
axes[0].tick_params(axis="x", rotation=20)

# Add legend for task types
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, edgecolor="black", label=t.replace("_", " "))
                   for t, c in type_colors.items()]
axes[0].legend(handles=legend_elements, fontsize=8, loc="upper left")

# Right: Class balance for binary classification tasks
bin_tasks = [t for t in tasks if t["task_type"] == "binary_classification"]
if bin_tasks:
    x_pos = np.arange(len(bin_tasks))
    pos_rates = [t["positive_rate"] for t in bin_tasks]
    neg_rates = [1 - r for r in pos_rates]
    names = [t["task_name"].split("/")[-1] for t in bin_tasks]
    
    axes[1].bar(x_pos, neg_rates, label="Negative", color="#56B4E9", edgecolor="black", linewidth=0.5)
    axes[1].bar(x_pos, pos_rates, bottom=neg_rates, label="Positive", color="#E69F00",
                edgecolor="black", linewidth=0.5)
    axes[1].set_xticks(x_pos)
    axes[1].set_xticklabels(names)
    axes[1].set_ylabel("Fraction")
    axes[1].set_title("Class Balance (Binary Classification Tasks)")
    axes[1].legend()
    
    for i, rate in enumerate(pos_rates):
        axes[1].text(i, 0.5, f"{rate:.1%}\npositive", ha="center", va="center", fontsize=9)

plt.tight_layout()
plt.show()

## 6. Unified Dataset Structure Summary

The transformation produces a unified JSON dataset where every item has a `type` field. This standardized format enables JRN experiments to process heterogeneous relational data (metadata, table contents, join statistics, task labels) through a single pipeline.

In [ ]:
# Final summary: dataset composition breakdown
print("=" * 60)
print("UNIFIED DATASET COMPOSITION")
print("=" * 60)
print(f"Total items in mini_demo_data.json: {len(data)}")
print()

for item_type, count in type_counts.most_common():
    print(f"  {item_type}: {count} item(s)")
    items_of_type = [it for it in data if it["type"] == item_type]
    for it in items_of_type:
        if item_type == "dataset_metadata":
            print(f"    → {it['dataset_name']}: {it['num_tables']} tables, "
                  f"{it['total_rows_across_tables']:,} total rows")
        elif item_type == "table_data":
            print(f"    → {it['table_name']}: {it['num_rows']:,} rows × {it['num_cols']} cols "
                  f"(sample: {len(it['data'])} rows)")
        elif item_type == "fk_join_metadata":
            print(f"    → {it['source_table']}.{it['source_fk_col']} → {it['target_table']}: "
                  f"{it['num_edges']:,} edges, coverage={it['join_coverage']:.1%}")
        elif item_type == "task_split":
            print(f"    → {it['task_name']} [{it['task_type']}]: "
                  f"{it['num_samples']:,} samples ({it['metadata_fold']})")

print()
print("Each item type maps to a specific JRN input/output format,")
print("enabling unified evaluation across relational database structures.")